In [0]:
%pip install pendulum

In [0]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import pendulum

In [0]:
%sql
--DROP DATABASE IF EXISTS workspace.silver CASCADE; 

In [0]:
yesterday = pendulum.now("America/Bogota").subtract(days=1)

path = f"/Volumes/workspace/bronze/uses/{yesterday.year}/{yesterday.month:02d}/{yesterday.day:02d}"

df_raw = spark.read.json(path)

In [0]:
print (path)

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.gold
COMMENT 'Capa gold de datos procesados'

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.uses_tx (
  id BIGINT,
  fechaCreacion TIMESTAMP,
  terminal_id BIGINT,
  tsn BIGINT,
  veh_id BIGINT,
  line_id BIGINT,
  app_id BIGINT,
  type_use BIGINT,
  fee BIGINT,
  uid BIGINT,
  trx_date TIMESTAMP,
  --flee_control STRING,
  state BIGINT,
  --state_description STRING,
  tp_Id BIGINT,
  --purseavalue DECIMAL(7,2),
  --pursebvalue DECIMAL(7,2),
  cut_id INT,
  errorcode BIGINT
)
USING DELTA
""")

In [0]:
query = f"select max(a.id) from workspace.gold.uses_tx a;"
max_id = spark.sql(query).first()[0]
print(max_id)


In [0]:
df_spark = spark.sql(f"""select id,
  fechaCreacion,
  terminal_id,
  tsn,
  veh_id,
  line_id,
  app_id,
  type_use,
  fee,
  uid,
  trx_date,
  state,
  tp_Id,
  cut_id,
  errorcode
from workspace.silver.uses a
where a.id > {max_id}""")
df = df_spark.toPandas()

In [0]:
display(df_spark.limit(5))

In [0]:
# Usamos append para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable("workspace.gold.uses_tx")



In [0]:
    display(df_spark.limit(5))